# CineMatch — Stage 2: Content-Based Recommender

In this stage, we implement the **Content-Based Filtering** recommendation model. 

### Core Concepts:
1. **TF-IDF (Term Frequency-Inverse Document Frequency)**: A statistical measure used to evaluate how important a word is to a document in a collection or corpus. In our case, the document is a combination of title, genres, and user tags, and the corpus is the collection of all movies or books.
2. **Cosine Similarity**: Measures the cosine of the angle between two vectors projected in a multi-dimensional space. The closer the cosine is to 1, the more similar the items.
3. **Sparse Representation**: Rather than storing a massive $N \times N$ dense similarity matrix in memory, we compute similarity vectors **on-the-fly** by calculating the dot product of a sparse target vector against our sparse TF-IDF matrix. This avoids scaling bottlenecks.

Let's import libraries and load our preprocessed data.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure our src files are in the python path
sys.path.append(os.path.abspath(".."))

from src.recommendation.content_based import ContentBasedRecommender

print("Recommender class successfully imported!")

Recommender class successfully imported!


## 1. Load Preprocessed Data

We load `movies_processed.csv` and `books_processed.csv` from the `data/processed` folder.

In [2]:
data_dir = "../data/processed"

movies_df = pd.read_csv(os.path.join(data_dir, "movies_processed.csv"))
books_df = pd.read_csv(os.path.join(data_dir, "books_processed.csv"))

print(f"Loaded {len(movies_df)} movies and {len(books_df)} books.")

Loaded 9742 movies and 10000 books.


## 2. Movies: Fit Recommender and Generate Recommendations

We fit the recommender on movies and fetch similar movies to a specific input target. Let's inspect the movies dataset to find a good target.

In [3]:
# Print a few popular movies to select an ID
print(movies_df[['movieId', 'title', 'genres']].head(10))

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   
5        6                         Heat (1995)   
6        7                      Sabrina (1995)   
7        8                 Tom and Huck (1995)   
8        9                 Sudden Death (1995)   
9       10                    GoldenEye (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
5                        Action|Crime|Thriller  
6                               Comedy|Romance  
7                           Adventure|Children  
8       

### 2.1 Fit Movie Recommender
We initialize and train our recommender using the movies dataset.

In [4]:
movie_recommender = ContentBasedRecommender(id_col='movieId', title_col='title', genre_col='genres')
movie_recommender.fit(movies_df)

# Check vocabulary size
vocab_size = len(movie_recommender.vectorizer.vocabulary_)
print(f"TF-IDF Vocabulary size: {vocab_size} words.")

TF-IDF Vocabulary size: 3703 words.


### 2.2 Recommend Similar Movies
Let's select **Toy Story (1995)**, which typically has `movieId = 1`, and recommend 10 similar items. We'll examine the structured evidence.

In [5]:
target_id = 1  # Toy Story
recs = movie_recommender.recommend_by_item(item_id=target_id, top_k=10)

print(f"=== RECOMMENDATIONS FOR TOY STORY ===\n")
for r in recs:
    print(f"Title: {r['title']}")
    print(f"Genres: {r['genres']}")
    print(f"Score: {r['score']:.4f}")
    print(f"Evidence: {r['evidence']}")
    print("-" * 50)

=== RECOMMENDATIONS FOR TOY STORY ===

Title: Toy Story 3 (2010)
Genres: Adventure|Animation|Children|Comedy|Fantasy|IMAX
Score: 0.6160
Evidence: {'content_score': 0.616, 'collaborative_score': 0.0, 'preference_score': 0.0, 'final_score': 0.616, 'matched_genres': ['Fantasy', 'Adventure', 'Animation', 'Children', 'Comedy'], 'similar_to': ['Toy Story (1995)']}
--------------------------------------------------
Title: Toy Story 2 (1999)
Genres: Adventure|Animation|Children|Comedy|Fantasy
Score: 0.5454
Evidence: {'content_score': 0.5454, 'collaborative_score': 0.0, 'preference_score': 0.0, 'final_score': 0.5454, 'matched_genres': ['Fantasy', 'Adventure', 'Animation', 'Children', 'Comedy'], 'similar_to': ['Toy Story (1995)']}
--------------------------------------------------
Title: Balto (1995)
Genres: Adventure|Animation|Children
Score: 0.4313
Evidence: {'content_score': 0.4313, 'collaborative_score': 0.0, 'preference_score': 0.0, 'final_score': 0.4313, 'matched_genres': ['Animation', 'Ch

### 2.3 User Profile Search
Let's query the movie recommendation engine using explicit structured user preferences.

In [6]:
user_prefs = {
    "genres": ["Sci-Fi", "Action"],
    "similar_to": ["Toy Story"],
    "keywords": ["space", "adventure"]
}

recs_profile = movie_recommender.recommend_by_profile(user_prefs, top_k=10)

print(f"=== PROFILE-BASED RECOMMENDATIONS ===\n")
for r in recs_profile:
    print(f"Title: {r['title']}")
    print(f"Genres: {r['genres']}")
    print(f"Score: {r['score']:.4f}")
    print(f"Evidence: {r['evidence']}")
    print("-" * 50)

=== PROFILE-BASED RECOMMENDATIONS ===

Title: Waterworld (1995)
Genres: Action|Adventure|Sci-Fi
Score: 0.4381
Evidence: {'content_score': 0.4381, 'collaborative_score': 0.0, 'preference_score': 0.4381, 'final_score': 0.4381, 'matched_genres': ['Action', 'Sci-Fi'], 'similar_to': ['Toy Story']}
--------------------------------------------------
Title: Balto (1995)
Genres: Adventure|Animation|Children
Score: 0.4343
Evidence: {'content_score': 0.4343, 'collaborative_score': 0.0, 'preference_score': 0.4343, 'final_score': 0.4343, 'matched_genres': [], 'similar_to': ['Toy Story']}
--------------------------------------------------
Title: The Space Between Us (2016)
Genres: Adventure|Sci-Fi
Score: 0.4204
Evidence: {'content_score': 0.4204, 'collaborative_score': 0.0, 'preference_score': 0.4204, 'final_score': 0.4204, 'matched_genres': ['Sci-Fi'], 'similar_to': ['Toy Story']}
--------------------------------------------------
Title: Ratchet & Clank (2016)
Genres: Action|Adventure|Animation|Chi

## 3. Books: Fit Recommender and Generate Recommendations

Now we apply the exact same architecture to books, showing its media-agnostic design.

In [7]:
# Print a few popular books
print(books_df[['book_id', 'title', 'authors', 'genres']].head(5))

   book_id                                              title  \
0        1            The Hunger Games (The Hunger Games, #1)   
1        2  Harry Potter and the Sorcerer's Stone (Harry P...   
2        3                            Twilight (Twilight, #1)   
3        4                              To Kill a Mockingbird   
4        5                                   The Great Gatsby   

                       authors                            genres  
0              Suzanne Collins  Classics|Romance|Fantasy|Mystery  
1  J.K. Rowling, Mary GrandPré    Sci-Fi|Romance|Fantasy|Mystery  
2              Stephenie Meyer          Classics|Fantasy|Mystery  
3                   Harper Lee                           Fiction  
4          F. Scott Fitzgerald          Classics|Fantasy|Mystery  


### 3.1 Fit Book Recommender

In [8]:
book_recommender = ContentBasedRecommender(id_col='book_id', title_col='title', genre_col='genres')
book_recommender.fit(books_df)
print(f"TF-IDF Vocabulary size (books): {len(book_recommender.vectorizer.vocabulary_)} words.")

TF-IDF Vocabulary size (books): 6357 words.


### 3.2 Recommend Similar Books
Let's select the first book in the dataset and recommend 5 similar items.

In [9]:
target_book_id = books_df.iloc[0]['book_id']
target_title = books_df.iloc[0]['title']
recs_books = book_recommender.recommend_by_item(item_id=target_book_id, top_k=5)

print(f"=== RECOMMENDATIONS FOR: '{target_title}' ===\n")
for r in recs_books:
    print(f"Title: {r['title']}")
    print(f"Genres: {r['genres']}")
    print(f"Score: {r['score']:.4f}")
    print(f"Evidence: {r['evidence']}")
    print("-" * 50)

=== RECOMMENDATIONS FOR: 'The Hunger Games (The Hunger Games, #1)' ===

Title: Mockingjay (The Hunger Games, #3)
Genres: Fiction
Score: 0.8838
Evidence: {'content_score': 0.8838, 'collaborative_score': 0.0, 'preference_score': 0.0, 'final_score': 0.8838, 'matched_genres': [], 'similar_to': ['The Hunger Games (The Hunger Games, #1)']}
--------------------------------------------------
Title: The Hunger Games Trilogy Boxset (The Hunger Games, #1-3)
Genres: Fiction
Score: 0.8305
Evidence: {'content_score': 0.8305, 'collaborative_score': 0.0, 'preference_score': 0.0, 'final_score': 0.8305, 'matched_genres': [], 'similar_to': ['The Hunger Games (The Hunger Games, #1)']}
--------------------------------------------------
Title: Catching Fire (The Hunger Games, #2)
Genres: Fiction
Score: 0.7523
Evidence: {'content_score': 0.7523, 'collaborative_score': 0.0, 'preference_score': 0.0, 'final_score': 0.7523, 'matched_genres': [], 'similar_to': ['The Hunger Games (The Hunger Games, #1)']}
--------

### 3.3 User Profile Search (Books)

In [10]:
book_prefs = {
    "genres": ["Fantasy"],
    "similar_to": [target_title],
    "keywords": ["magic", "wizard"]
}

recs_book_profile = book_recommender.recommend_by_profile(book_prefs, top_k=5)

print(f"=== PROFILE-BASED BOOK RECOMMENDATIONS ===\n")
for r in recs_book_profile:
    print(f"Title: {r['title']}")
    print(f"Genres: {r['genres']}")
    print(f"Score: {r['score']:.4f}")
    print(f"Evidence: {r['evidence']}")
    print("-" * 50)

=== PROFILE-BASED BOOK RECOMMENDATIONS ===

Title: Off to Be the Wizard (Magic 2.0, #1)
Genres: Non-Fiction|Historical Fiction
Score: 0.5377
Evidence: {'content_score': 0.5377, 'collaborative_score': 0.0, 'preference_score': 0.5377, 'final_score': 0.5377, 'matched_genres': [], 'similar_to': ['The Hunger Games (The Hunger Games, #1)']}
--------------------------------------------------
Title: Wizard at Large (Magic Kingdom of Landover, #3)
Genres: Fiction
Score: 0.4490
Evidence: {'content_score': 0.449, 'collaborative_score': 0.0, 'preference_score': 0.449, 'final_score': 0.449, 'matched_genres': [], 'similar_to': ['The Hunger Games (The Hunger Games, #1)']}
--------------------------------------------------
Title: Witch & Wizard (Witch & Wizard, #1)
Genres: Fiction
Score: 0.4277
Evidence: {'content_score': 0.4277, 'collaborative_score': 0.0, 'preference_score': 0.4277, 'final_score': 0.4277, 'matched_genres': [], 'similar_to': ['The Hunger Games (The Hunger Games, #1)']}
--------------

C:\Users\ASUS\.gemini\antigravity\scratch\cinematch\src\recommendation\content_based.py:136: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  self.items_df[self.title_col].str.contains(item_name, case=False, na=False)


Stage 2 Content-Based Recommendation is fully complete! We have built a sparse representation TF-IDF recommender that works on-the-fly and supports both movies and books natively, producing rich structured evidence. We are ready to proceed to Stage 3: Collaborative Filtering!